In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    median_absolute_error,
    explained_variance_score,
)
import category_encoders as ce
from lightgbm import LGBMRegressor, early_stopping

try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as exc:
    raise ImportError(
        "optuna is required for this RMSE-optimized pipeline. Install with: pip install optuna"
    ) from exc


# =========================
# CONFIG
# =========================
DATA_DIR = '/kaggle/input/competitions/mlx-2-0-regression'
TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH = f'{DATA_DIR}/test.csv'
SUBMISSION_PATH = 'submission_lgb_optimized.csv'

RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 120
EARLY_STOPPING_ROUNDS = 200
MAX_ESTIMATORS = 6000
ENSEMBLE_SEEDS = [42, 2024, 7]


def add_calendar_features(df):
    df = df.copy()
    df['publication_timestamp'] = pd.to_datetime(df['publication_timestamp'])
    df['year'] = df['publication_timestamp'].dt.year
    df['month'] = df['publication_timestamp'].dt.month
    df['day'] = df['publication_timestamp'].dt.day
    df['dayofweek'] = df['publication_timestamp'].dt.dayofweek
    df['is_month_start'] = df['publication_timestamp'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['publication_timestamp'].dt.is_month_end.astype(int)
    df.drop('publication_timestamp', axis=1, inplace=True)
    return df


def add_features(df):
    df = df.copy()

    groups = [
        'emotional_charge',
        'duration_ms',
        'intensity_index',
        'rhythmic_cohesion',
        'groove_efficiency',
        'instrumental_density',
        'organic_texture',
        'organic_immersion',
    ]

    for g in groups:
        cols = [f'{g}_0', f'{g}_1', f'{g}_2']
        if all(col in df.columns for col in cols):
            values = df[cols]
            df[f'{g}_mean'] = values.mean(axis=1)
            df[f'{g}_std'] = values.std(axis=1)
            df[f'{g}_range'] = values.max(axis=1) - values.min(axis=1)

    if {'beat_frequency_0', 'beat_frequency_1'}.issubset(df.columns):
        df['tempo_ratio_10'] = df['beat_frequency_1'] / (df['beat_frequency_0'] + 1e-6)
        df['tempo_delta_10'] = df['beat_frequency_1'] - df['beat_frequency_0']

    if {'intensity_index_0', 'intensity_index_1', 'intensity_index_2'}.issubset(df.columns):
        df['energy_sum_012'] = (
            df['intensity_index_0'] + df['intensity_index_1'] + df['intensity_index_2']
        )

    if 'month' in df.columns:
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

    if 'dayofweek' in df.columns:
        df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7.0)
        df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7.0)

    return df


def preprocess_with_fit(X_fit, y_fit, X_apply_list):
    X_fit = X_fit.copy()
    X_apply_list = [df.copy() for df in X_apply_list]

    title_cols = [
        col for col in ['composition_label_0', 'composition_label_1', 'composition_label_2']
        if col in X_fit.columns
    ]

    for col in title_cols:
        freq = X_fit[col].value_counts()
        X_fit[col] = X_fit[col].map(freq)
        for df in X_apply_list:
            df[col] = df[col].map(freq)

    if title_cols:
        X_fit[title_cols] = X_fit[title_cols].fillna(0)
        for df in X_apply_list:
            df[title_cols] = df[title_cols].fillna(0)

    num_cols = X_fit.select_dtypes(include=['number', 'bool']).columns
    cat_cols = X_fit.select_dtypes(include=['object']).columns

    median_vals = X_fit[num_cols].median()
    X_fit[num_cols] = X_fit[num_cols].fillna(median_vals)
    for df in X_apply_list:
        df[num_cols] = df[num_cols].fillna(median_vals)

    X_fit[cat_cols] = X_fit[cat_cols].fillna('Unknown')
    for df in X_apply_list:
        df[cat_cols] = df[cat_cols].fillna('Unknown')

    low_card_cols = [col for col in cat_cols if X_fit[col].nunique(dropna=False) < 12]
    high_card_cols = [col for col in cat_cols if X_fit[col].nunique(dropna=False) >= 12]

    if low_card_cols:
        ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_fit[low_card_cols] = ord_encoder.fit_transform(X_fit[low_card_cols])
        for df in X_apply_list:
            df[low_card_cols] = ord_encoder.transform(df[low_card_cols])

    if high_card_cols:
        target_encoder = ce.TargetEncoder(cols=high_card_cols, smoothing=30)
        X_fit[high_card_cols] = target_encoder.fit_transform(X_fit[high_card_cols], y_fit)
        for df in X_apply_list:
            df[high_card_cols] = target_encoder.transform(df[high_card_cols])

    X_fit = add_features(X_fit)
    X_apply_list = [add_features(df) for df in X_apply_list]

    clean_cols = X_fit.columns.str.replace('[^A-Za-z0-9_]', '_', regex=True)
    X_fit.columns = clean_cols

    aligned_apply = []
    for df in X_apply_list:
        df = df.set_axis(df.columns.str.replace('[^A-Za-z0-9_]', '_', regex=True), axis=1)
        df = df.reindex(columns=X_fit.columns, fill_value=0.0)
        aligned_apply.append(df.astype(np.float32))

    X_fit = X_fit.astype(np.float32)
    return (X_fit, *aligned_apply)


def safe_mape(y_true, y_pred, eps=1e-8):
    y_true_np = np.asarray(y_true)
    y_pred_np = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true_np), eps)
    return float(np.mean(np.abs((y_true_np - y_pred_np) / denom)))


def smape(y_true, y_pred, eps=1e-8):
    y_true_np = np.asarray(y_true)
    y_pred_np = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true_np) + np.abs(y_pred_np), eps)
    return float(np.mean(2.0 * np.abs(y_pred_np - y_true_np) / denom))


def wape(y_true, y_pred, eps=1e-8):
    y_true_np = np.asarray(y_true)
    y_pred_np = np.asarray(y_pred)
    return float(np.sum(np.abs(y_true_np - y_pred_np)) / max(np.sum(np.abs(y_true_np)), eps))


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'mse': float(mse),
        'rmse': float(np.sqrt(mse)),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'medae': float(median_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'explained_variance': float(explained_variance_score(y_true, y_pred)),
        'mape': safe_mape(y_true, y_pred),
        'smape': smape(y_true, y_pred),
        'wape': wape(y_true, y_pred),
    }


def run_lgb_cv(params, X_raw, y, n_splits=5, base_seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=base_seed)
    oof_preds = np.zeros(len(X_raw), dtype=np.float64)
    best_iterations = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_raw), start=1):
        X_tr_raw = X_raw.iloc[train_idx].copy()
        y_tr = y.iloc[train_idx].copy()
        X_val_raw = X_raw.iloc[val_idx].copy()
        y_val = y.iloc[val_idx].copy()

        X_tr, X_val = preprocess_with_fit(X_tr_raw, y_tr, [X_val_raw])

        model = LGBMRegressor(
            **params,
            objective='regression',
            n_estimators=MAX_ESTIMATORS,
            force_col_wise=True,
            n_jobs=-1,
            random_state=base_seed + fold,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric='rmse',
            callbacks=[early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)],
        )

        current_best = model.best_iteration_ if model.best_iteration_ is not None else MAX_ESTIMATORS
        best_iterations.append(int(current_best))
        oof_preds[val_idx] = model.predict(X_val, num_iteration=current_best)

    metrics = compute_metrics(y, oof_preds)
    return metrics, oof_preds, best_iterations


# =========================
# LOAD DATA
# =========================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train.drop('track_identifier', axis=1, inplace=True, errors='ignore')
test.drop('track_identifier', axis=1, inplace=True, errors='ignore')

train = add_calendar_features(train)
test = add_calendar_features(test)

X_full_raw = train.drop(['target', 'id'], axis=1).copy()
y = train['target'].copy()
X_test_raw = test.drop(['id'], axis=1).copy()


# =========================
# OPTUNA: RMSE OPTIMIZATION
# =========================
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.08, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 24, 256),
        'max_depth': trial.suggest_int('max_depth', 4, 14),
        'min_child_samples': trial.suggest_int('min_child_samples', 8, 150),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.2),
    }

    fold_metrics, _, _ = run_lgb_cv(
        params=params,
        X_raw=X_full_raw,
        y=y,
        n_splits=N_SPLITS,
        base_seed=RANDOM_STATE,
    )
    return fold_metrics['rmse']


study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

best_params = study.best_trial.params
print('Best Optuna RMSE:', study.best_value)
print('Best Params:', best_params)


# =========================
# FINAL CV EVALUATION
# =========================
cv_metrics, _, best_iterations = run_lgb_cv(
    params=best_params,
    X_raw=X_full_raw,
    y=y,
    n_splits=N_SPLITS,
    base_seed=RANDOM_STATE,
)

print('CV MSE:', cv_metrics['mse'])
print('CV RMSE:', cv_metrics['rmse'])
print('CV MAE:', cv_metrics['mae'])
print('CV MedAE:', cv_metrics['medae'])
print('CV R2:', cv_metrics['r2'])
print('CV Explained Variance:', cv_metrics['explained_variance'])
print('CV MAPE:', cv_metrics['mape'])
print('CV sMAPE:', cv_metrics['smape'])
print('CV WAPE:', cv_metrics['wape'])


# =========================
# FINAL TRAINING + ENSEMBLE
# =========================
X_full, X_test_final = preprocess_with_fit(X_full_raw, y, [X_test_raw])
final_n_estimators = max(int(np.median(best_iterations)), 300)

test_preds = np.zeros(len(X_test_final), dtype=np.float64)
for seed in ENSEMBLE_SEEDS:
    final_model = LGBMRegressor(
        **best_params,
        objective='regression',
        n_estimators=final_n_estimators,
        force_col_wise=True,
        n_jobs=-1,
        random_state=seed,
    )
    final_model.fit(X_full, y)
    test_preds += final_model.predict(X_test_final)

test_preds /= len(ENSEMBLE_SEEDS)


# =========================
# SAVE SUBMISSION
# =========================
submission = pd.DataFrame(
    {
        'id': test['id'],
        'target': test_preds,
    }
)
submission.to_csv(SUBMISSION_PATH, index=False)
print('Saved:', SUBMISSION_PATH)
